# Run a fine-tuned Gemma 3 12B (LoRA adapter)

Loads the base model in 4-bit, attaches your saved adapter, and lets you chat with it.
Run the cells top to bottom **once** in a fresh kernel

In [1]:
# --- Settings: edit these ---
BASE_MODEL   = "google/gemma-3-12b-it"
ADAPTER_PATH = "gemma3-12b-gsm8k-es/final"      # folder containing adapter_model.safetensors
# HF_CACHE   = "/path/to/big/storage/hf_cache"   # uncomment if your home quota is small

import os
# if "HF_CACHE" in globals(): os.environ["HF_HOME"] = HF_CACHE   # must run before importing transformers
print("Adapter folder exists:", os.path.isdir(ADAPTER_PATH), "->", os.path.abspath(ADAPTER_PATH))
print("Contents:", os.listdir(ADAPTER_PATH) if os.path.isdir(ADAPTER_PATH) else "NOT FOUND")

Adapter folder exists: True -> /workspace/NLP-and-LLMs-Math-by-Language-Project/fine_tuning/gemma3-12b-gsm8k-es/final
Contents: ['README.md', 'adapter_config.json', 'tokenizer_config.json', 'chat_template.jinja', 'adapter_model.safetensors', '.ipynb_checkpoints', 'tokenizer.json', 'training_args.bin']


In [8]:
# Only needed if you haven't logged in on this machine before
# from huggingface_hub import login
# login()

In [2]:
import os
print("cwd:", os.getcwd())
for root, dirs, files in os.walk("/path/to/your/project"):   # e.g. the NLP-and-LLMs-Math-by-Language-Project folder
    if "adapter_config.json" in files:
        print(root)

cwd: /workspace/NLP-and-LLMs-Math-by-Language-Project/fine_tuning


In [3]:
import os
root = "/workspace/NLP-and-LLMs-Math-by-Language-Project"
found = False
for dirpath, dirs, files in os.walk(root):
    if "adapter_config.json" in files:
        print("FOUND:", dirpath)
        found = True
if not found:
    print("No adapter_config.json anywhere under", root)

print("\nFiles next to this notebook:", os.listdir("."))

FOUND: /workspace/NLP-and-LLMs-Math-by-Language-Project/fine_tuning/gemma3-12b-lora/checkpoint-125
FOUND: /workspace/NLP-and-LLMs-Math-by-Language-Project/fine_tuning/gemma3-12b-gsm8k-es/checkpoint-16
FOUND: /workspace/NLP-and-LLMs-Math-by-Language-Project/fine_tuning/gemma3-12b-gsm8k-es/final

Files next to this notebook: ['gemma3-12b-lora', 'run_finetuned.ipynb', 'spark_testing.ipynb', 'gemma3-12b-gsm8k-es', '.ipynb_checkpoints', 'train_gsm8k_per_language.ipynb']


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

# Attach the fine-tuned adapter
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("Loaded base model + adapter")

Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

Loaded base model + adapter


In [5]:
def chat(messages, max_new_tokens=300, temperature=0.7, top_p=0.9):
    """messages: list of {"role": "user"/"assistant", "content": "..."} (alternating, starting with user)."""
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            top_p=top_p if temperature > 0 else None,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]   # only the reply, not the prompt
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print(chat([{"role": "user", "content": "Explain LoRA in two sentences."}]))

LoRA (Low-Rank Adaptation) is a technique that fine-tunes large language models by only training a small number of extra parameters, significantly reducing computational cost and storage space. It achieves this by approximating weight updates with low-rank matrices, allowing for efficient adaptation to new tasks while preserving the original model's knowledge.


## Interactive chat (keeps history)
Type `quit` to stop.

In [ ]:
history = []
while True:
    user = input("You: ")
    if user.strip().lower() in {"quit", "exit"}:
        break
    history.append({"role": "user", "content": user})
    reply = chat(history)
    history.append({"role": "assistant", "content": reply})
    print("Model:", reply, "\n")

You:  Mike juega al ping-pong durante 40 minutos. En los primeros 20 minutos, anota 4 puntos. En los siguientes 20 minutos, anota un 25 % más de puntos. ¿Cuántos puntos anotó en total?


Model: En los primeros 20 minutos, anotó 4 puntos. En los siguientes 20 minutos, anotó un 25 % más de puntos, lo que significa que anotó 4 * 0.25 = 1 punto más. En total, anotó 4 + 1 = 5 puntos. #### 5 



You:  John planea vender todos sus juguetes y usar el dinero para comprar videojuegos. Tiene 13 sets de Lego y los vende a $15 cada uno. Termina comprando 8 videojuegos a $20 cada uno y le sobran $5. ¿Cuántos sets de Lego le quedan?


Model: John vende sus sets de Lego por 13 * $15 = $195. Compró 8 videojuegos a $20 cada uno, por lo que gastó $20 * 8 = $160 en videojuegos. Después de comprar videojuegos, John le quedaron $195 - $160 = $35. Después de gastar $35 en videojuegos, John tenía $35 - $5 = $30. Por lo tanto, John tenía $30 en sets de Lego restantes. #### 30 



## Optional: compare against the base model
Temporarily disables the adapter so you can see what fine-tuning changed.

In [ ]:
prompt = [{"role": "user", "content": "Your test question here"}]

print("FINE-TUNED:\n", chat(prompt), "\n")
with model.disable_adapter():
    print("BASE:\n", chat(prompt))

## Optional: merge adapter into a standalone model
Needs enough memory to load the base in bf16 (~24 GB), so **restart the kernel first** and skip the 4-bit cells above.

In [ ]:
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from peft import PeftModel
#
# base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.bfloat16, device_map="auto")
# merged = PeftModel.from_pretrained(base, ADAPTER_PATH).merge_and_unload()
# merged.save_pretrained("gemma3-12b-merged")
# AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained("gemma3-12b-merged")